# Proceso de ingesta de datos para guardar en la capa bronce

La capa bronce es la primera etapa en la arquitectura medallion del lakehouse. Aquí se almacenan los datos en su estado original, sin validaciones ni transformaciones, permitiendo conservar la fidelidad y trazabilidad de la fuente.

**Pasos del proceso de ingesta:**

1. **Obtención de datos:** Los datos históricos desde el WMS Manhattan WMOS se almacenan en Azure Blob Storage, se obtienen desde Blob Storage y se guardan los datos en el Datalake manteniendo una capa de datos en bruto (raw).sistemas federados.
2. **Ingesta incremental:** Se utiliza una herramienta como Auto Loader para detectar y procesar nuevos archivos a medida que llegan, permitiendo escalabilidad y manejo eficiente de grandes volúmenes.
3. **Almacenamiento en bronce:** Los datos se guardan en la capa bronce en su formato original (por ejemplo, JSON, CSV, Avro), sin modificaciones.
4. **Retención histórica:** Se conserva todo el historial de datos, lo que facilita auditorías y reprocesamientos futuros.

Esta capa sirve como fuente de verdad para posteriores enriquecimientos y transformaciones en las capas silver y gold.

## 1. _Configuración_

In [0]:
# Celda 1: Recepción de Parámetros
table_name = dbutils.widgets.text("table_name", "")
table_name = dbutils.widgets.get("table_name")

base_path = dbutils.widgets.text("base_path", "")
base_path = dbutils.widgets.get("base_path")

file_name = dbutils.widgets.text("file_name", "")
file_name = dbutils.widgets.get("file_name")

schema_name = dbutils.widgets.text("schema_name", "")
schema_name = dbutils.widgets.get("schema_name")

In [0]:
# Librerias de errores
from pyspark.errors import AnalysisException

In [0]:
# Crear el Esquema si no existe
spark.sql(f"CREATE DATABASE IF NOT EXISTS `{schema_name}`")

## 2. Cargar el conjunto de `datos`

In [0]:
# Se define la ruta completa del archivo de entrada y el nombre de la tabla de salida
input_path = f"{base_path}/{file_name}"
output_table_name = f"{schema_name}.{table_name}_raw"
print(f"input_path: {input_path}")
print(f"output_table_name: {output_table_name}")

In [0]:

# Leemos el archivo CSV, indicando que tiene encabezado y que infiera el esquema
df_raw = (
    spark.read
         .option("header", True)         # si tiene header
         .option("inferSchema", True)    # infiere tipos
         .csv(input_path)
)

## 3. Guardar el conjunto de `datos` en capa bronce

In [0]:
# Escribimos el DataFrame en formato Delta y lo registramos como tabla
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table_name)

In [0]:
# Verifica conteos y muestra esquema
try:
    df_check = spark.table(output_table_name)
    print("Filas Bronze:", df_check.count())
    df_check.printSchema()
    display(df_check.limit(10))
except AnalysisException as e:
    print("Bronze aún no materializado (si estás en streaming, espera al menos 1 microbatch).", str(e))